In [21]:
import pandas as pd
import numpy as np

df_raw = pd.read_csv('rds_data.csv')

print(df_raw.head())
print(df_raw.shape)


   Sl.No Ref No Data  Entry code   IP No    Birth Term  \
0      1           1          98  583504  Preterm        
1      2           2          99  583602  Preterm        
2      3           3         100  583603  Preterm        
3      4           4         101  583894  Preterm        
4      5           5         102  584019  Preterm        

  Gestational Age (Weeks) Category    Place  Gender  Type  ...  \
0                      29      SGA  Inborn     Male  LSCS  ...   
1                      32      SGA  Inborn   Female  LSCS  ...   
2                      32      AGA  Inborn   Female  LSCS  ...   
3                      34      AGA  Inborn     Male  LSCS  ...   
4                      28      SGA  Inborn     Male  LSCS  ...   

  Head circumference  Length Mother Age Mother Hb Mother Blood Group  \
0             N/A        NaN         31       NaN                NaN   
1         29.5 cm        NaN    29 yrs        NaN         O Positive   
2              30 cm     NaN    29 yrs

In [22]:

row_count = df_raw.shape[0]
missing_pct = (df_raw.isna().sum() / row_count).sort_values(ascending=False)
print(missing_pct.head(15))

cat_cols = ['Birth Term','Category','Place','Gender','Type','Presentation','Mother Blood Group','Parity','Target']
for col in cat_cols:
    if col in df_raw.columns:
        vals = (df_raw[col].astype('string')
                .str.strip()
                .str.replace(r'\s+', ' ', regex=True)
                .str.lower())
        print(col)
        print(vals.value_counts(dropna=False).head(10))

print(df_raw.columns)

Unnamed: 15                 0.997078
Other Complications/Info    0.853178
Length                      0.550767
Mother Hb                   0.444120
Head circumference          0.254200
Gestational Age (Weeks)     0.192841
Mother Age                  0.168736
Arterial Blood Po2          0.157049
Arterial Blood Pco2         0.153397
Arterial Blood Ph           0.149744
SpO2                        0.136596
Presentation                0.102264
Mother Blood Group          0.099343
Apgar at 1                  0.092038
Apgar at 5                  0.091308
dtype: float64
Birth Term
Birth Term
preterm            433
term               413
late preterm       272
early preterm      104
extreme preterm     53
<NA>                22
-                   18
late pretrem        10
extrem preterm       8
late-preterm         6
Name: count, dtype: Int64
Category
Category
aga                                    1002
sga                                     191
<NA>                                     51
- 

In [23]:
import re

df = df_raw.copy()

# 1) Standardize column names
col_ren = {}
for col in df.columns:
    col_clean = str(col)
    col_clean = col_clean.replace('\u00a0', ' ')
    col_clean = re.sub(r'\s+', ' ', col_clean).strip()
    col_ren[col] = col_clean

df = df.rename(columns=col_ren)

# Drop almost-empty column
if 'Unnamed: 15' in df.columns:
    df = df.drop(columns=['Unnamed: 15'])

# 2) Normalize all object/string cells
obj_cols = df.select_dtypes(include=['object']).columns.tolist()
for col in obj_cols:
    df[col] = (df[col].astype('string')
               .str.replace(r'\s+', ' ', regex=True)
               .str.strip())

# Treat common missing tokens as NA
missing_tokens = {'-', '—', '_', 'na', 'n/a', 'nil', 'none', ''}
for col in obj_cols:
    df[col] = df[col].where(~df[col].str.lower().isin(missing_tokens), pd.NA)

# 3) Harmonize key categorical columns
# Place
if 'Place' in df.columns:
    place_norm = df['Place'].str.lower()
    place_norm = place_norm.replace({
        'in born': 'inborn',
        'inborn ': 'inborn',
        'out born': 'outborn',
        'born': pd.NA,
        'new born': pd.NA,
        'nicu': pd.NA
    })
    df['Place'] = place_norm

# Gender
if 'Gender' in df.columns:
    gender_norm = df['Gender'].str.lower().replace({
        'feamle': 'female',
        'femae': 'female',
        'make': 'male',
        'female & male': pd.NA
    })
    df['Gender'] = gender_norm

# Type (mode of delivery)
if 'Type' in df.columns:
    type_norm = df['Type'].str.lower().replace({
        'normal vaginal delivery': 'nvd',
        'vd': 'nvd',
        'vad': 'nvd',
        'lcls': 'lscs',
        'emergency lscs': 'lscs'
    })
    df['Type'] = type_norm

# Presentation
if 'Presentation' in df.columns:
    pres_norm = df['Presentation'].str.lower().replace({
        'cephalic': 'vertex'
    })
    # keep compound/rare presentations as-is; normalize whitespace already done
    df['Presentation'] = pres_norm

# Birth term
if 'Birth Term' in df.columns:
    bt_norm = df['Birth Term'].str.lower()
    bt_norm = bt_norm.replace({
        'late pretrem': 'late preterm',
        'late-preterm': 'late preterm',
        'extrem preterm': 'extreme preterm'
    })
    df['Birth Term'] = bt_norm

# Target
if 'Target' in df.columns:
    df['Target'] = df['Target'].str.lower().replace({'pos': 'positive', 'neg': 'negative'})

# 4) Parse numeric columns from text

def extract_first_float(val):
    if pd.isna(val):
        return np.nan
    txt = str(val)
    m = re.search(r'-?\d+(?:\.\d+)?', txt)
    if not m:
        return np.nan
    return float(m.group(0))

num_like_cols = [
    'Gestational Age (Weeks)', 'Birth weight (gms)', 'Apgar at 1', 'Apgar at 5',
    'Head circumference', 'Length', 'Mother Age', 'Mother Hb'
]
for col in num_like_cols:
    if col in df.columns:
        df[col] = df[col].apply(extract_first_float)

# ABG columns (ph, pco2, po2)
abg_cols = ['Arterial Blood Ph', 'Arterial Blood Pco2', 'Arterial Blood Po2']
# Handle trailing spaces in original names by matching startswith
for col in list(df.columns):
    col_l = col.lower().strip()
    if col_l.startswith('arterial blood ph'):
        df = df.rename(columns={col: 'Arterial Blood Ph'})
    if col_l.startswith('arterial blood pco2'):
        df = df.rename(columns={col: 'Arterial Blood Pco2'})
    if col_l.startswith('arterial blood po2'):
        df = df.rename(columns={col: 'Arterial Blood Po2'})

for col in ['Arterial Blood Ph','Arterial Blood Pco2','Arterial Blood Po2']:
    if col in df.columns:
        df[col] = df[col].apply(extract_first_float)

# Heart rate, respiratory rate, SpO2 can contain text; extract numeric features
if 'Heart rate' in df.columns:
    df['Heart_rate_bpm'] = df['Heart rate'].apply(extract_first_float)
if 'Respiratory rate' in df.columns:
    df['Resp_rate_cpm'] = df['Respiratory rate'].apply(extract_first_float)
if 'SpO2' in df.columns:
    df['SpO2_pct'] = df['SpO2'].apply(extract_first_float)

# 5) Basic sanity filters (set extreme impossible values to NA)
if 'Gestational Age (Weeks)' in df.columns:
    df.loc[(df['Gestational Age (Weeks)'] < 20) | (df['Gestational Age (Weeks)'] > 45), 'Gestational Age (Weeks)'] = np.nan
if 'Birth weight (gms)' in df.columns:
    df.loc[(df['Birth weight (gms)'] < 300) | (df['Birth weight (gms)'] > 7000), 'Birth weight (gms)'] = np.nan
if 'SpO2_pct' in df.columns:
    df.loc[(df['SpO2_pct'] < 10) | (df['SpO2_pct'] > 100), 'SpO2_pct'] = np.nan

print(df.head())
print(df.shape)

   Sl.No Ref No Data  Entry code   IP No Birth Term  Gestational Age (Weeks)  \
0      1           1          98  583504    preterm                     29.0   
1      2           2          99  583602    preterm                     32.0   
2      3           3         100  583603    preterm                     32.0   
3      4           4         101  583894    preterm                     34.0   
4      5           5         102  584019    preterm                     28.0   

  Category   Place  Gender  Type  ... Mother Hb Mother Blood Group    Parity  \
0      SGA  inborn    male  lscs  ...       NaN               <NA>      G2P1   
1      SGA  inborn  female  lscs  ...       NaN         O Positive  G2 L0 A1   
2      AGA  inborn  female  lscs  ...       NaN         O Positive  G2 L0 A2   
3      AGA  inborn    male  lscs  ...       NaN         O Positive  G2 P2 L2   
4      SGA  inborn    male  lscs  ...       NaN         A Positive  G2 P0 A1   

   Arterial Blood Ph  Arterial Blood P

C:\Users\gakow\AppData\Local\Temp\ipykernel_18736\3333459375.py:20: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  obj_cols = df.select_dtypes(include=['object']).columns.tolist()


In [24]:
df.to_csv('rds_data_clean.csv', index=False)

In [25]:
for column in df.columns:
    unique_values = df[column].unique()
    print(f"{column}: {unique_values}")


Sl.No: [   1    2    3 ... 1367 1368 1369]
Ref No Data: <StringArray>
[  '1',   '2',   '3',   '4',   '5',   '6',   '7',   '8',   '9',  '11',
 ...
 '394', '395', '396', '397', '398', '399', '400', '401', '402', '403']
Length: 364, dtype: string
Entry code: [       98        99       100 ... 230048179 230048917 230048971]
IP No: <StringArray>
['583504', '583602', '583603', '583894', '584019', '584164', '584605',
 '584711', '586182', '586423',
 ...
 '937439', '937440', '937441', '937442', '937443', '937444', '937445',
 '937446', '937447', '937448']
Length: 1259, dtype: string
Birth Term: <StringArray>
[            'preterm',                'term',        'late preterm',
                  <NA>,   'remove(above age)', 'extreme prematurity',
     'extreme preterm',       'early preterm',     'extream preterm',
               'terms',         'latepreterm',           'late term',
       '(no template)',           'post term',      'exterm preterm',
           'full term',   'extremely preterm

In [26]:
df.columns

Index(['Sl.No', 'Ref No Data', 'Entry code', 'IP No', 'Birth Term',
       'Gestational Age (Weeks)', 'Category', 'Place', 'Gender', 'Type',
       'Presentation', 'Other Complications/Info', 'Birth weight (gms)',
       'Apgar at 1', 'Apgar at 5', 'Heart rate', 'Respiratory rate', 'SpO2',
       'Head circumference', 'Length', 'Mother Age', 'Mother Hb',
       'Mother Blood Group', 'Parity', 'Arterial Blood Ph',
       'Arterial Blood Pco2', 'Arterial Blood Po2', 'Target', 'Heart_rate_bpm',
       'Resp_rate_cpm', 'SpO2_pct'],
      dtype='str')

In [28]:
df=df.drop(['Sl.No', 'Ref No Data', 'Entry code', 'IP No'], axis=1)
df

,Birth Term,Gestational Age (Weeks),Category,Place,Gender,Type,Presentation,Other Complications/Info,Birth weight (gms),Apgar at 1,...,Mother Hb,Mother Blood Group,Parity,Arterial Blood Ph,Arterial Blood Pco2,Arterial Blood Po2,Target,Heart_rate_bpm,Resp_rate_cpm,SpO2_pct
0,preterm,29.0,SGA,inborn,male,lscs,vertex,<NA>,880.0,3.0,...,NaN,<NA>,G2P1,7.21,35.0,55.0,positive,20.0,NaN,NaN
1,preterm,32.0,SGA,inborn,female,lscs,breech,<NA>,1095.0,6.0,...,NaN,O Positive,G2 L0 A1,7.42,25.0,33.0,positive,132.0,62.0,65.0
2,preterm,32.0,AGA,inborn,female,lscs,breech,<NA>,1500.0,8.0,...,NaN,O Positive,G2 L0 A2,NaN,35.0,50.0,positive,148.0,48.0,98.0
3,preterm,34.0,AGA,inborn,male,lscs,breech,<NA>,2045.0,8.0,...,NaN,O Positive,G2 P2 L2,7.29,26.0,37.0,positive,150.0,66.0,80.0
4,preterm,28.0,SGA,inborn,male,lscs,vertex,<NA>,800.0,1.0,...,NaN,A Positive,G2 P0 A1,7.23,33.0,32.0,positive,126.0,38.0,92.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1364,term,NaN,AGA,outborn,male,lscs,<NA>,<NA>,3155.0,8.0,...,NaN,B Negative,G3P1L2A1,NaN,NaN,NaN,negative,168.0,78.0,88.0
1365,term,NaN,AGA,outborn,male,nvd,<NA>,<NA>,2430.0,8.0,...,NaN,B Positive,G2A1,7.12,78.0,32.0,negative,134.0,65.0,88.0
1366,preterm,NaN,AGA,outborn,male,nvd,<NA>,<NA>,2680.0,NaN,...,NaN,B Positive,G2A1,7.21,59.0,71.0,negative,152.0,66.0,98.0
1367,term,36.0,SGA,inborn,male,lscs,vertex,<NA>,2950.0,8.0,...,NaN,O Positive,Primigravida,7.21,60.0,42.0,negative,132.0,50.0,97.0


In [29]:
df.columns

Index(['Birth Term', 'Gestational Age (Weeks)', 'Category', 'Place', 'Gender',
       'Type', 'Presentation', 'Other Complications/Info',
       'Birth weight (gms)', 'Apgar at 1', 'Apgar at 5', 'Heart rate',
       'Respiratory rate', 'SpO2', 'Head circumference', 'Length',
       'Mother Age', 'Mother Hb', 'Mother Blood Group', 'Parity',
       'Arterial Blood Ph', 'Arterial Blood Pco2', 'Arterial Blood Po2',
       'Target', 'Heart_rate_bpm', 'Resp_rate_cpm', 'SpO2_pct'],
      dtype='str')

In [30]:
df.isnull().sum()

Birth Term                    44
Gestational Age (Weeks)      346
Category                      89
Place                        181
Gender                        57
Type                          87
Presentation                 212
Other Complications/Info    1170
Birth weight (gms)           118
Apgar at 1                   218
Apgar at 5                   218
Heart rate                    51
Respiratory rate              60
SpO2                         299
Head circumference           509
Length                      1105
Mother Age                   325
Mother Hb                    866
Mother Blood Group           216
Parity                       146
Arterial Blood Ph            312
Arterial Blood Pco2          317
Arterial Blood Po2           321
Target                         0
Heart_rate_bpm                52
Resp_rate_cpm                 70
SpO2_pct                     387
dtype: int64

In [31]:
df=df.drop(['Heart rate', 'Respiratory rate', 'SpO2', 'Other Complications/Info'], axis=1)
df.columns

Index(['Birth Term', 'Gestational Age (Weeks)', 'Category', 'Place', 'Gender',
       'Type', 'Presentation', 'Birth weight (gms)', 'Apgar at 1',
       'Apgar at 5', 'Head circumference', 'Length', 'Mother Age', 'Mother Hb',
       'Mother Blood Group', 'Parity', 'Arterial Blood Ph',
       'Arterial Blood Pco2', 'Arterial Blood Po2', 'Target', 'Heart_rate_bpm',
       'Resp_rate_cpm', 'SpO2_pct'],
      dtype='str')

In [32]:
df.isnull().sum()

Birth Term                   44
Gestational Age (Weeks)     346
Category                     89
Place                       181
Gender                       57
Type                         87
Presentation                212
Birth weight (gms)          118
Apgar at 1                  218
Apgar at 5                  218
Head circumference          509
Length                     1105
Mother Age                  325
Mother Hb                   866
Mother Blood Group          216
Parity                      146
Arterial Blood Ph           312
Arterial Blood Pco2         317
Arterial Blood Po2          321
Target                        0
Heart_rate_bpm               52
Resp_rate_cpm                70
SpO2_pct                    387
dtype: int64

In [33]:
print("---UNIQUE VALUES IN EACH COLUMN---")
for column in df.columns:
    unique_values = df[column].unique()
    print(f"{column}: {unique_values}")


---UNIQUE VALUES IN EACH COLUMN---
Birth Term: <StringArray>
[            'preterm',                'term',        'late preterm',
                  <NA>,   'remove(above age)', 'extreme prematurity',
     'extreme preterm',       'early preterm',     'extream preterm',
               'terms',         'latepreterm',           'late term',
       '(no template)',           'post term',      'exterm preterm',
           'full term',   'extremely preterm',           'earlyterm',
      'exteme preterm',    'moderate preterm',        'septic shock',
       'late pre-term',       'late pre term']
Length: 23, dtype: string
Gestational Age (Weeks): [29. 32. 34. 28. 30. nan 33. 24. 36. 26. 31. 25. 37. 39. 35. 40. 38. 27.
 22. 41. 21.]
Category: <StringArray>
[                                                 'SGA',
                                                  'AGA',
                                                   <NA>,
                     'RDS, Sepsis, Twin, ASD, Zone III',
            